#### Step 1 导入相关包

In [ ]:
import pandas as pd
import numpy as np
import json
import re
import torch
import pickle
import json

from tqdm import tqdm
from collections import defaultdict
import pickle
from itertools import combinations

import networkx as nx
# The version you see now is the original one. After several revisions, the final version is different. 
# The code is kept at school, and we will update it in March during the holiday break. (2026.2.7)

#### Step 2 读取pickle和txt数据

In [ ]:
# 打开文件
with open('../id2_affiliation_entity_id.pickle', 'rb') as file:
    # 读取文件内容
    id2_affiliation_entity = pickle.load(file)

In [ ]:
# 打开文件
with open('../data/zscore_dic.pickle', 'rb') as file:
    # 输入文件内容
    zscore_dict = pickle.load(file)

In [ ]:
df_normalized = pd.read_parquet('../data/normalized-ents.parquet')
df_normalized

In [ ]:
# 打开文件
with open('../ent-type.txt', 'r') as file:
    # 读取文件内容
    file_contents = file.read()
id_type_json = json.loads(file_contents)

#### Step 3 构建共现关系

In [ ]:
# 创建一个 defaultdict 用于存储关键词共现次数
per_year_entity = defaultdict(dict)
for year in range(2000, 2023):
    per_year_entity[year] = defaultdict(list)
    for institution_type in ['academic','industry','cooperation']:
        per_year_entity[year][institution_type] = []

for paper_id in tqdm(id2_affiliation_entity):
    year = id2_affiliation_entity[paper_id]['year']
    cooperation_type = id2_affiliation_entity[paper_id]['Cooperation type']
    ent_list = id2_affiliation_entity[paper_id]['entity']

    temp_lists = []
    # 合并四种实体类型数据
    for entity_type in ['Method','Dataset','Tool','Metric']:
        temp_lists.extend(id2_affiliation_entity[paper_id]['entity'][entity_type])
    try:
        per_year_entity[year][cooperation_type].append(temp_lists)
    except Exception as e:
        print(e)

In [ ]:
with open('../data/per_year_entity.pickle', 'wb') as file:
    pickle.dump(per_year_entity,file)

In [ ]:
per_year_occurrence = defaultdict(dict)
for year in range(2000, 2023):
    per_year_occurrence[year] = defaultdict(dict)
    
# edge的值大于等于2
for year in tqdm(per_year_entity):
    co_occurrence = defaultdict(int)
    for institution_type in ['industry','academic','cooperation']:
        for entity_list in per_year_entity[year][institution_type]:
            for keyword1, keyword2 in combinations(entity_list, 2):
                co_occurrence[(keyword1, keyword2)] += 1
                
    per_year_occurrence[year]= {key:value for key,value in co_occurrence.items() if value>=2}

In [ ]:
graph_list = defaultdict(dict)
for year in tqdm(per_year_occurrence):
    # 创建一个无向图
    G = nx.Graph()
    # 遍历字典中的每一个键值对，添加边到图中
    for (node1, node2), weight in per_year_occurrence[year].items():
        G.add_node(node1)
        G.add_node(node2)

        # 添加双向边，并将权重加起来
        if G.has_edge(node1, node2):
            G[node1][node2]['weight'] += weight
        else:
            G.add_edge(node1, node2, weight=weight)
    graph_list[year] = G

In [ ]:
with open('../data/graph comp/total_graph_list.pickle', 'wb') as file:
    pickle.dump(graph_list,file)

In [ ]:
# 计算最大连通子图
lists1 = []
for year in range(2000,2023):
    largest_bb = max(nx.connected_components(graph_list[year]), key=len)
    lists1.append(len(largest_bb))
    # 输出最大连通子图的规模
    print(f"{year} 最大连通子图的规模:{len(largest_bb)}" )

#### Step 4 分析子图

In [ ]:
per_year_sub_occurrence = defaultdict(dict)
for year in range(2000, 2023):
    per_year_sub_occurrence[year] = defaultdict(dict)
    

for year in tqdm(per_year_entity):
    for institution_type in ['industry','academic','cooperation']:
        co_occurrence = defaultdict(int)
        for entity_list in per_year_entity[year][institution_type]:
            for keyword1, keyword2 in combinations(entity_list, 2):
                co_occurrence[(keyword1, keyword2)] += 1
                
        per_year_sub_occurrence[year][institution_type]= {key:value for key,value in co_occurrence.items() if value>=2}

In [ ]:
sub_graph_list = defaultdict(dict)
for year in tqdm(per_year_sub_occurrence):
    sub_graph_list[year] = defaultdict(dict)
    for institution_type in ['industry','academic','cooperation']:
        # 创建一个无向图
        G = nx.Graph()
        # 遍历字典中的每一个键值对，添加边到图中
        for (node1, node2), weight in per_year_sub_occurrence[year][institution_type].items():
            G.add_node(node1)
            G.add_node(node2)

            # 添加双向边，并将权重加起来
            if G.has_edge(node1, node2):
                G[node1][node2]['weight'] += weight
            else:
                G.add_edge(node1, node2, weight=weight)
        sub_graph_list[year][institution_type] = G

In [ ]:
import matplotlib.pyplot as plt
from networkx import Graph

def weighted_jaccard_index(edges1, edges2):
    # 创建权重字典
    weights1 = {edge: data.get('weight', 1) for edge, data in edges1.items()}
    weights2 = {edge: data.get('weight', 1) for edge, data in edges2.items()}

    # 转换为集合
    set1 = set(weights1.keys())
    set2 = set(weights2.keys())

    # 计算加权交集和并集
    intersection = set1.intersection(set2)
    union = set1.union(set2)
    
    intersection_weight = sum(min(weights1[edge], weights2[edge]) for edge in intersection)
    union_weight = sum(max(weights1.get(edge, 0), weights2.get(edge, 0)) for edge in union)
    
    return intersection_weight / union_weight if union_weight != 0 else 0

# 初始化用于存储杰卡德系数和边数量的字典
jaccard_data = {
    'industry_academic': [],
    'industry_cooperation': [],
    'academic_cooperation': []
}
edges_count = {
    'academic': [],
    'industry': [],
    'cooperation': []
}

years = list(range(2000, 2023))

for year in years:
    # 提取每个图类型
    graphs = {t: sub_graph_list[year][t] for t in ['industry', 'academic', 'cooperation']}
    
    # 找到三个图的公共节点
    common_nodes = set(graphs['industry'].nodes()).intersection(set(graphs['academic'].nodes())).intersection(set(graphs['cooperation'].nodes()))
    
    # 创建公共节点的子图
    subgraph_industry = graphs['industry'].subgraph(common_nodes)
    subgraph_academic = graphs['academic'].subgraph(common_nodes)
    subgraph_cooperation = graphs['cooperation'].subgraph(common_nodes)
    
    # 手动构建边的字典
    edges_industry = {(u, v): d for u, v, d in subgraph_industry.edges(data=True)}
    edges_academic = {(u, v): d for u, v, d in subgraph_academic.edges(data=True)}
    edges_cooperation = {(u, v): d for u, v, d in subgraph_cooperation.edges(data=True)}
    
    # 计算加权杰卡德系数
    weighted_jaccard_industry_academic = weighted_jaccard_index(edges_industry, edges_academic)
    weighted_jaccard_industry_cooperation = weighted_jaccard_index(edges_industry, edges_cooperation)
    weighted_jaccard_academic_cooperation = weighted_jaccard_index(edges_academic, edges_cooperation)
    
    # 记录边数量
    edges_count['academic'].append(len(list(subgraph_academic.edges())))
    edges_count['industry'].append(len(list(subgraph_industry.edges())))
    edges_count['cooperation'].append(len(list(subgraph_cooperation.edges())))

    # 将结果保存到字典中
    jaccard_data['industry_academic'].append(weighted_jaccard_industry_academic)
    jaccard_data['industry_cooperation'].append(weighted_jaccard_industry_cooperation)
    jaccard_data['academic_cooperation'].append(weighted_jaccard_academic_cooperation)

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# 设置全局字体
plt.rcParams['font.family'] = 'Times New Roman'

years = list(range(2000, 2023))

fig, ax1 = plt.subplots(figsize=(16,8), dpi=300)

# 创建主坐标轴，绘制边数量的柱状图
bar_width = 0.25
x = list(range(len(years)))

# 绘制学术图的边数量
ax1.bar([p - bar_width for p in x], edges_count['academic'], width=bar_width, label='Academic Edges', color='royalblue', alpha=0.9, edgecolor='black')
# 绘制工业图的边数量
ax1.bar([p + bar_width for p in x], edges_count['industry'], width=bar_width, label='Industry Edges', color='forestgreen', alpha=0.9, edgecolor='black')
# 绘制合作图的边数量
ax1.bar(x, edges_count['cooperation'], width=bar_width, label='Cooperation Edges', color='tomato', alpha=0.9, edgecolor='black')

# 设置主坐标轴的属性
ax1.set_xlabel('Year', fontsize=25)
ax1.set_ylabel('Number of Edges', fontsize=25)

# 设置主图图例的位置
ax1.legend(loc='center', bbox_to_anchor=(0.5, 0.95), ncol=3, frameon=True, fontsize=15)

ax1.grid(axis='y', linestyle='--', alpha=0.7)

# 设置x轴的刻度和标签
ax1.set_xticks(x)
ax1.set_xticklabels(years, rotation=45)

# 设置 x 轴和 y 轴刻度的字体大小
ax1.tick_params(axis='x', labelsize=15)  # x 轴刻度的字体大小设置为 15
ax1.tick_params(axis='y', labelsize=15)  # y 轴刻度的字体大小设置为 15

# 通过 bbox_to_anchor 设置子图的相对位置和大小
ax2 = inset_axes(ax1, width='50%', height='50%', bbox_to_anchor=(0.05, 0.2, 1, 1), bbox_transform=ax1.transAxes, loc='lower left')

# 绘制折线图
ax2.plot(years, jaccard_data['industry_academic'], 
         marker='o', linestyle='-', linewidth=2, color='#1f77b4', 
         label='Industry-Academic')
ax2.plot(years, jaccard_data['industry_cooperation'], 
         marker='s', linestyle='--', linewidth=2, color='#ff7f0e', 
         label='Industry-Cooperation')
ax2.plot(years, jaccard_data['academic_cooperation'], 
         marker='^', linestyle='-.', linewidth=2, color='#2ca02c', 
         label='Academic-Cooperation')

# 添加2013的虚线
ax2.axvline(x=2013, color='gray', linestyle='--', linewidth=1.5)

# 填充2013和2022之间的区域
ax2.fill_betweenx(y=[min(jaccard_data['industry_academic'] + jaccard_data['industry_cooperation'] + jaccard_data['academic_cooperation']),
                      max(jaccard_data['industry_academic'] + jaccard_data['industry_cooperation'] + jaccard_data['academic_cooperation'])],
                  x1=2013, x2=2022, color='yellow', alpha=0.3)

# 设置 x 轴和 y 轴刻度的字体大小
ax2.tick_params(axis='x', labelsize=12)  # x 轴刻度的字体大小设置为 12
ax2.tick_params(axis='y', labelsize=12)  # y 轴刻度的字体大小设置为 12

# 设置子图的属性
ax2.set_ylabel('Jaccard Index')
ax2.legend(loc='center left', bbox_to_anchor=(-0.1, -0.15), ncol=3, frameon=True, fontsize=15)
ax2.grid(True)

plt.show()

In [ ]:
import matplotlib.pyplot as plt

# 设置全局字体
plt.rcParams['font.family'] = 'Times New Roman'

years = list(range(2000, 2023))

# 创建柱状图
fig1, ax1 = plt.subplots(figsize=(16, 8), dpi=300)

# 创建主坐标轴，绘制边数量的柱状图
bar_width = 0.25
x = list(range(len(years)))

# 绘制学术图的边数量
ax1.bar([p - bar_width for p in x], edges_count['academic'], width=bar_width, label='Academic Edges', color='royalblue', alpha=0.9, edgecolor='black')
# 绘制工业图的边数量
ax1.bar([p + bar_width for p in x], edges_count['industry'], width=bar_width, label='Industry Edges', color='forestgreen', alpha=0.9, edgecolor='black')
# 绘制合作图的边数量
ax1.bar(x, edges_count['cooperation'], width=bar_width, label='Cooperation Edges', color='tomato', alpha=0.9, edgecolor='black')

# 设置主坐标轴的属性
ax1.set_xlabel('Year', fontsize=25)
ax1.set_ylabel('Number of Edges', fontsize=25)

# 设置主图图例的位置
ax1.legend(loc='center', bbox_to_anchor=(0.5, 0.95), ncol=3, frameon=True, fontsize=18)

ax1.grid(axis='y', linestyle='--', alpha=0.7)

# 设置x轴的刻度和标签
ax1.set_xticks(x)
ax1.set_xticklabels(years, rotation=45)

# 设置 x 轴和 y 轴刻度的字体大小
ax1.tick_params(axis='x', labelsize=20)  # x 轴刻度的字体大小设置为 15
ax1.tick_params(axis='y', labelsize=20)  # y 轴刻度的字体大小设置为 15
# 调整图形布局以适应标题和标签
fig1.tight_layout(rect=[0, 0.0, 1, 1])
# 显示柱状图
plt.show()

# 创建折线图
fig2, ax2 = plt.subplots(figsize=(16, 8), dpi=300)  # 保持与柱状图相同的尺寸

# 绘制折线图
ax2.plot(years, jaccard_data['industry_academic'],
         marker='o', linestyle='-', linewidth=2, color='#1f77b4',
         label='Industry-Academic')
ax2.plot(years, jaccard_data['industry_cooperation'],
         marker='s', linestyle='--', linewidth=2, color='#ff7f0e',
         label='Industry-Cooperation')
ax2.plot(years, jaccard_data['academic_cooperation'],
         marker='^', linestyle='-.', linewidth=2, color='#2ca02c',
         label='Academic-Cooperation')

# 添加2013的虚线
ax2.axvline(x=2013, color='gray', linestyle='--', linewidth=1.5)


ax2.set_ylim(0, 0.45)  # 例如，设置 x 轴范围从 2010 到 2025

y_min = ax2.get_ylim()[0]  # 获取 y 轴的最小值
y_max = ax2.get_ylim()[1]  # 获取 y 轴的最大值

# 填充2013和2022之间的区域
ax2.fill_between(x=[2013, 2022],
                 y1=y_min,
                 y2=y_max,
                 color='goldenrod', alpha=0.3)


# 设置 x 轴和 y 轴刻度的字体大小与柱状图一致
ax2.tick_params(axis='x', labelsize=20)  # x 轴刻度的字体大小设置为 15
ax2.tick_params(axis='y', labelsize=20)  # y 轴刻度的字体大小设置为 15

# 设置子图的属性
ax2.set_ylabel('Jaccard Index', fontsize=25)
ax2.set_xlabel('Year', fontsize=25)  # 可选：添加 x 轴标签

# 设置 x 轴刻度为每年一个刻度
ax2.set_xticks(years)  # 将刻度设置为每一年
ax2.set_xticklabels(years, rotation=45)

ax2.legend(loc='center left', bbox_to_anchor=(0.2, 0.95), ncol=3, frameon=True, fontsize=18)
ax2.grid(False)
# 调整图形布局以适应标题和标签
fig2.tight_layout(rect=[0, 0.0, 1, 1])
# 显示折线图
plt.show()

In [ ]:
sub_graph_list[2000]['industry'].subgraph([149,0,2,25]).edges()

In [ ]:
# 打开文件
with open('../data/graph comp/sub_graph_list.pickle', 'wb') as file:
    # 读取文件内容
    pickle.dump(sub_graph_list,file)

#### Step 5 分析合并的图

In [ ]:
per_year_merged_sub = defaultdict(dict)
for year in range(2000, 2023):
    per_year_merged_sub[year] = defaultdict(dict)

for year in tqdm(per_year_entity):
    co_occurrence = defaultdict(int)
    for institution_type in ['industry', 'academic']:
        for entity_list in per_year_entity[year][institution_type]:
            for keyword1, keyword2 in combinations(entity_list, 2):
                co_occurrence[(keyword1, keyword2)] += 1
                
    co_occurrence_values = list(co_occurrence.values())
    
    per_year_merged_sub[year] = {key: value for key, value in co_occurrence.items() if value >= 2}

In [ ]:
sub_graph_merged_list = defaultdict(dict)
for year in tqdm(per_year_merged_sub):
    sub_graph_merged_list[year] = defaultdict(dict)
    # 创建一个无向图
    G = nx.Graph()
    # 遍历字典中的每一个键值对，添加边到图中
    for (node1, node2), weight in per_year_merged_sub[year].items():
        G.add_node(node1)
        G.add_node(node2)

        # 添加双向边，并将权重加起来
        if G.has_edge(node1, node2):
            G[node1][node2]['weight'] += weight
        else:
            G.add_edge(node1, node2, weight=weight)
    sub_graph_merged_list[year] = G

In [ ]:
sub_graph_list[2000]['industry'].nodes()

In [ ]:
for year in range(2000, 2023):
    node_set = sub_graph_list[year]['industry'].nodes()
    edges_in_subset = [(u, v) for u, v in sub_graph_merged_list[year].edges() if u in node_set and v in node_set]
    total_num_edges = len(edges_in_subset)
    edges = len(sub_graph_list[year]['industry'].edges())
    ratio = 1 - edges/total_num_edges

    print(f"Year {year}: Number of edges between nodes in the set: {ratio}")

In [ ]:
per_year_sub_occurrence_2 = defaultdict(dict)
for year in range(2000, 2023):
    per_year_sub_occurrence_2[year] = defaultdict(dict)
    

for year in tqdm(per_year_entity):
    for institution_type in ['industry','academic','cooperation']:
        co_occurrence = defaultdict(int)
        for entity_list in per_year_entity[year][institution_type]:
            for keyword1, keyword2 in combinations(entity_list, 2):
                co_occurrence[(keyword1, keyword2)] += 1
                
        per_year_sub_occurrence_2[year][institution_type]= {key:value for key,value in co_occurrence.items() if value>=2}